# ML-04 - Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shaheerkhalid04/Shaheer-Khalid-FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

ML-03 said my lane is a ranking task with a forward-window label. This notebook writes down what a row actually means on the warehouse, and then checks every line of it with a query instead of trusting my own sentence.

I work on **`month=2026-03`** as the development month and leave June 2026 alone. The final month is the natural outcome window of any past-to-future label, so developing there means developing inside my own test set.

The headline: one column I fully intended to use turned out to be unusable, and I only found it because a feature came back negative. That's section 2.

In [1]:
# Setup. Token order: env var -> Colab secret -> local credential store.
# Never paste a token into a cell, this repo is public.
import os
import time

import duckdb
import numpy as np
import pandas as pd

while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
    os.chdir("..")

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

con = duckdb.connect()
con.execute("SET enable_progress_bar=false")
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
else:
    # Falls back to the machine's stored HF login, so no token appears in this file.
    con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, PROVIDER credential_chain)")

REL = "hf://datasets/FlyRank/internship-warehouse"
CUTOFF = "2026-03-31"

MONTH_03 = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
FEAT_MONTHS = "read_parquet([" + ", ".join(
    f"'{REL}/fact_content_daily_performance/month=2026-{m}/*.parquet'" for m in ("01", "02", "03")
) + "])"
LABEL_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"

print(f"connected. cutoff T = {CUTOFF}")

connected. cutoff T = 2026-03-31


## 1. Unit of analysis + time window

**The contract, in plain words. Answers 1 to 3 of 5.**

**1. One row = one page, observed at one cutoff date T.** Concretely: one `content_hash_id`, belonging to one `client_hash_id`, summarised as it stood on **2026-03-31**. Not a page-day, not a page-query. The daily fact table is the page-day grain, and I roll it up to the page. The editor opens a page, so the page is what I score.

**2. Which tables.**

| Table | What I take from it |
|---|---|
| `fact_content_daily_performance` | everything that carries a `report_date`, which is every number I trust |
| `dim_content` | `content_type` only, as context, for reasons section 2 explains |
| `dim_clients` | `gsc_data_start` and access flags, to size the panel limitation |

I do **not** touch `fact_content_query_90d` here, and that's answer 5.

**3. The time windows.** Two windows that never overlap, which is the whole point:

```text
feature window   2026-01-01 .. 2026-03-31   (90 days, everything on or before T)
                 |
            T = 2026-03-31
                 |
label window     2026-04-01 .. 2026-04-30   (the next 30 days, never a feature)
```

April is still mid-panel, so nothing here goes near the sealed June test month.

The cell below proves the grain claim rather than asserting it: if one row really is one page-day, then grouping by `(report_date, client_hash_id, content_hash_id)` can never return a group of more than one.

In [2]:
# QUERY 1 of 3 - the grain. Empty result = one row really is one page-day.
t = time.time()
grain = con.sql(f'''
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MONTH_03}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
''').df()

print(f"duplicate page-days found: {len(grain)}   [{time.time() - t:.0f}s]")
print("0 means the grain claim holds, so rolling up to one row per page is safe.")
grain

duplicate page-days found: 0   [22s]
0 means the grain claim holds, so rolling up to one row per page is safe.


,report_date,client_hash_id,content_hash_id,c


## 2. Fields: feature / label / context / excluded

**Answers 4 and 5 of 5.**

**4. What I predict.** For a page with at least 50 impressions in March, does its April search demand fall more than 20% below March?

```text
y = 1 if impressions[2026-04-01 .. 2026-04-30] < 0.8 x impressions[2026-03-01 .. 2026-03-31]
```

Observed, not defined. Nobody wrote this into a column, it's measured by comparing two windows I cut myself. That's the fix for the problem I found in ML-03, where the starter file's label was a rule restated.

**5. What I deliberately exclude: `last_optimized_date`, `content_updated_date`, and anything derived from them, including the staleness feature I had planned.**

This is the part I got wrong first, so I'm showing the work. In ML-03 staleness was the single best rule I tested, so my plan was to make "days since last optimised" feature number five. I built it, and it came back **negative**: minus 83, minus 86, minus 44 days. A page optimised 83 days *after* the moment I'm supposed to be predicting from.

The query below explains why. `last_optimized_date` has 45,396 non-null values in `dim_content`, and the **earliest one is 2026-04-24**. Every single one lands after my cutoff. `dim_content` is a **current-state dimension**, not a snapshot one: it describes each page as it stood when the release was exported in July 2026, with no history. `content_updated_date` has the same disease, 382,739 rows sitting after T.

So the exclusion rule I'm adopting for the whole capstone: **a `dim_content` column is only safe if it cannot change after the page is created.** Dates that get rewritten when someone edits a page are future information wearing a factual-looking name.

The four buckets:

| Bucket | Fields |
|---|---|
| **Feature** | `imp_90d`, `momentum`, `avg_position_m0`, `active_days_m0`, `ctr_m0`, all built only from `report_date <= T` |
| **Label** | April impressions, and `y` derived from them. Never a feature |
| **Context** | `client_hash_id` (grouping and splitting), `content_hash_id` (joins), `content_type` (reading results) |
| **Excluded** | `last_optimized_date`, `content_updated_date` (dated after T, shown below); `fact_content_query_90d` (its fixed 90-day window overlaps my April label period, so its `impressions_90d` columns contain the answer); `gsc_sum_position` raw (kept only as the input to a weighted average) |

In [3]:
# Why last_optimized_date is excluded: every non-null value lands AFTER my cutoff.
dates = con.sql(f'''
    SELECT
        COUNT(*)                                                                    AS content_rows,
        SUM(CASE WHEN last_optimized_date IS NULL THEN 1 ELSE 0 END)                AS optimized_null,
        SUM(CASE WHEN last_optimized_date > DATE '{CUTOFF}' THEN 1 ELSE 0 END)      AS optimized_after_T,
        MIN(last_optimized_date)                                                    AS earliest_optimized,
        SUM(CASE WHEN content_updated_date > DATE '{CUTOFF}' THEN 1 ELSE 0 END)     AS updated_after_T
    FROM {DIM_CONTENT}
''').df()

print(dates.to_string(index=False))
print()
non_null = int(dates.content_rows[0] - dates.optimized_null[0])
print(f"non-null last_optimized_date values: {non_null:,}")
print(f"of those, dated after T:             {int(dates.optimized_after_T[0]):,}")
print(f"earliest one:                        {dates.earliest_optimized[0]}  (T is {CUTOFF})")
print()
print("so there is no value of this column I could have known at T. it is future information.")
print("excluded, along with content_updated_date and the staleness feature I meant to build.")

 content_rows  optimized_null  optimized_after_T earliest_optimized  updated_after_T
       519606        474210.0            45396.0         2026-04-24         382739.0

non-null last_optimized_date values: 45,396
of those, dated after T:             45,396
earliest one:                        2026-04-24 00:00:00  (T is 2026-03-31)

so there is no value of this column I could have known at T. it is future information.
excluded, along with content_updated_date and the staleness feature I meant to build.


## 3. Verify it with queries, then five features, then the trap

### Queries 2 and 3 of 3

Query 1 was the grain. Query 2 is the size and span of my slice. Query 3 is availability, and it is the one with a real trap in it.

The availability flags are **three-valued**: TRUE, FALSE, and NULL. That means `WHERE ga4_data_available = FALSE` and `WHERE NOT ga4_data_available` are not the same question, because NULL is neither. In March 2026 that gap is **3,018,741 rows**, about 31% of the month. Writing `= FALSE` silently throws them away and never tells you. So every availability filter I write uses `IS TRUE` or `IS NOT TRUE`, which handle NULL the way I mean.

In [4]:
# QUERY 2 of 3 - how big is my slice and what does it span?
t = time.time()
span = con.sql(f'''
    SELECT
        COUNT(*)                        AS page_days,
        COUNT(DISTINCT content_hash_id) AS pages,
        COUNT(DISTINCT client_hash_id)  AS clients,
        MIN(report_date)                AS first_day,
        MAX(report_date)                AS last_day
    FROM {MONTH_03}
''').df()
print(span.to_string(index=False))
print(f"[{time.time() - t:.0f}s]")

 page_days  pages  clients  first_day   last_day
   9841378 331437       55 2026-03-01 2026-03-31
[2s]


In [5]:
# QUERY 3 of 3 - availability, filtered with IS TRUE, and why '= FALSE' is a trap.
t = time.time()
avail = con.sql(f'''
    SELECT
        COUNT(*)                                                             AS all_rows,
        CAST(SUM(CASE WHEN gsc_data_available IS TRUE  THEN 1 ELSE 0 END) AS BIGINT) AS gsc_is_true,
        CAST(SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END) AS BIGINT) AS ga4_is_true,
        CAST(SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS BIGINT) AS ga4_is_false,
        CAST(SUM(CASE WHEN ga4_data_available IS NULL  THEN 1 ELSE 0 END) AS BIGINT) AS ga4_is_null,
        CAST(SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) AS BIGINT) AS ga4_is_not_true
    FROM {MONTH_03}
''').df()

row = avail.iloc[0]
print(f"rows in month=2026-03:            {row.all_rows:>12,}")
print(f"gsc_data_available IS TRUE:       {row.gsc_is_true:>12,}  <- the rows I keep")
print(f"ga4_data_available IS TRUE:       {row.ga4_is_true:>12,}")
print()
print("the three-valued trap, same column, two ways of asking 'not available':")
print(f"  ... IS NOT TRUE  ->             {row.ga4_is_not_true:>12,}")
print(f"  ... IS FALSE     ->             {row.ga4_is_false:>12,}")
print(f"  difference (the NULLs):         {row.ga4_is_null:>12,}  ({row.ga4_is_null / row.all_rows * 100:.0f}% of the month)")
print()
print(f"[{time.time() - t:.0f}s]  writing '= FALSE' would drop those rows without a word.")

rows in month=2026-03:               9,841,378
gsc_data_available IS TRUE:          3,611,061  <- the rows I keep
ga4_data_available IS TRUE:            413,966

the three-valued trap, same column, two ways of asking 'not available':
  ... IS NOT TRUE  ->                9,427,412
  ... IS FALSE     ->                6,408,671
  difference (the NULLs):            3,018,741  (31% of the month)

[9s]  writing '= FALSE' would drop those rows without a word.


### The five features

Five, no more. Each one is built only from rows with `report_date <= T`, and each gets the one line that matters: **why is this knowable at the decision moment?**

| # | Feature | Knowable at the decision moment because... |
|---|---|---|
| 1 | `imp_90d` | it sums impressions over 2026-01-01 to 2026-03-31, every day of which is on or before T |
| 2 | `momentum` | it divides March impressions by February impressions, two windows that both close on or before T |
| 3 | `avg_position_m0` | it is March's impression-weighted average position, computed from March rows only |
| 4 | `active_days_m0` | it counts March days with at least one impression, and March is complete at T |
| 5 | `ctr_m0` | it is March clicks over March impressions, both measured before T |

Every one comes from `fact_content_daily_performance`, because that table carries a `report_date` and so I can prove when each number was true. That is exactly what `dim_content` could not give me.

The pull below is the heavy cell, roughly 90 seconds over the network, and it caches to `work/outputs/` so re-running costs nothing. The data skill is blunt about this: repeated full scans hit HTTP 429, so I scan once and keep the result.

In [6]:
# Build the feature frame. One scan, then cached.
CACHE = "work/outputs/w03_feature_frame.parquet"

if os.path.exists(CACHE):
    frame = pd.read_parquet(CACHE)
    print(f"loaded cached frame: {len(frame):,} rows")
else:
    t = time.time()
    frame = con.sql(f'''
        WITH feat AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions)                                                          AS imp_90d,
                SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_impressions ELSE 0 END) AS imp_m0,
                SUM(CASE WHEN report_date BETWEEN DATE '2026-02-01' AND DATE '2026-02-28'
                         THEN gsc_impressions ELSE 0 END)                                     AS imp_m1,
                SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_clicks ELSE 0 END)      AS clk_m0,
                SUM(CASE WHEN report_date >= DATE '2026-03-01' THEN gsc_sum_position ELSE 0 END) AS pos_sum_m0,
                COUNT(DISTINCT CASE WHEN report_date >= DATE '2026-03-01' AND gsc_impressions > 0
                                    THEN report_date END)                                     AS active_days_m0
            FROM {FEAT_MONTHS}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        ),
        lab AS (
            SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_next30
            FROM {LABEL_MONTH}
            WHERE gsc_data_available IS TRUE
            GROUP BY 1, 2
        )
        SELECT f.*, d.content_type, COALESCE(l.imp_next30, 0) AS imp_next30
        FROM feat f
        LEFT JOIN lab l USING (client_hash_id, content_hash_id)
        LEFT JOIN {DIM_CONTENT} d USING (client_hash_id, content_hash_id)
        WHERE f.imp_m0 >= 50
    ''').df()
    os.makedirs("work/outputs", exist_ok=True)
    frame.to_parquet(CACHE, index=False)
    print(f"pulled {len(frame):,} rows in {time.time() - t:.0f}s, cached to {CACHE}")

# DuckDB scans in parallel and does not promise row order, so two identical pulls can
# come back in different orders and shift the group split. Sort once, deterministically.
frame = frame.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)

# The two derived features, and the label.
frame["momentum"] = frame["imp_m0"] / frame["imp_m1"].replace(0, np.nan)
frame["ctr_m0"] = frame["clk_m0"] / frame["imp_m0"] * 100
frame["avg_position_m0"] = frame["pos_sum_m0"] / frame["imp_m0"].replace(0, np.nan)
frame["y"] = (frame["imp_next30"] < 0.8 * frame["imp_m0"]).astype(int)

FEATURES = ["imp_90d", "momentum", "avg_position_m0", "active_days_m0", "ctr_m0"]
model_frame = frame.dropna(subset=FEATURES).copy()

print(f"eligible pages (>=50 March impressions): {len(frame):,}")
print(f"usable after dropping null features:     {len(model_frame):,}")
print(f"clients represented:                     {model_frame['client_hash_id'].nunique()}")
print(f"base rate of April demand loss:          {model_frame['y'].mean() * 100:.1f}%")
print()
model_frame[["content_hash_id"] + FEATURES + ["y"]].head()

pulled 116,114 rows in 85s, cached to work/outputs/w03_feature_frame.parquet
eligible pages (>=50 March impressions): 116,114
usable after dropping null features:     97,967
clients represented:                     39
base rate of April demand loss:          55.4%



,content_hash_id,imp_90d,momentum,avg_position_m0,active_days_m0,ctr_m0,y
0,content_04c67f3541177192,858.0,1.345528,14.377644,31,0.60423,0
1,content_0f30e04e709c7b5d,377.0,1.198347,8.124138,30,0.00000,1
2,content_1207efddce873942,967.0,2.649425,14.488069,31,0.00000,0
3,content_167472cd0802a8f3,724.0,1.414634,11.961207,29,0.00000,0
4,content_27f8100281413b37,654.0,7.733333,9.437500,18,0.00000,0


### The trap, performed on purpose

Now the leak. I add **one** column that is derived from the label window, `imp_next30`, the April impressions the label is computed from. I do not tell the model it's special. Then I watch what happens.

The split is `GroupShuffleSplit` on `client_hash_id`, so no client appears in both train and test. That's the split I committed to in ML-03, and it matters here: a random row split would let the model recognise a client it has already seen and inflate both numbers.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(model_frame, model_frame["y"],
                                          groups=model_frame["client_hash_id"]))
train, test = model_frame.iloc[train_idx], model_frame.iloc[test_idx]


def score(columns, tag):
    model = RandomForestClassifier(n_estimators=200, random_state=42,
                                   n_jobs=-1, min_samples_leaf=5)
    model.fit(train[columns], train["y"])
    probs = model.predict_proba(test[columns])[:, 1]
    auc = roc_auc_score(test["y"], probs)
    top200 = test.assign(_p=probs).nlargest(200, "_p")["y"].mean()
    print(f"{tag:34s} AUC={auc:.4f}   precision@200={top200:.3f}")
    return auc, model


print(f"train: {train['client_hash_id'].nunique()} clients / {len(train):,} pages")
print(f"test:  {test['client_hash_id'].nunique()} clients / {len(test):,} pages")
print(f"test base rate: {test['y'].mean():.3f}")
print()

honest_auc, honest_model = score(FEATURES, "honest, 5 features")

# The deliberate leak: April impressions, which the label is computed from.
model_frame["leak_imp_next30"] = model_frame["imp_next30"]
train, test = model_frame.iloc[train_idx], model_frame.iloc[test_idx]
leaked_auc, leaked_model = score(FEATURES + ["leak_imp_next30"], "WITH label-derived column")

print()
print(f"the leak bought {leaked_auc - honest_auc:+.4f} AUC and none of it is real.")
print()
print("what the leaked model leaned on:")
for name, importance in sorted(zip(FEATURES + ["leak_imp_next30"],
                                   leaked_model.feature_importances_), key=lambda p: -p[1]):
    flag = "  <- the leak" if name == "leak_imp_next30" else ""
    print(f"  {name:20s} {importance:.3f}{flag}")

train: 29 clients / 93,421 pages
test:  10 clients / 4,546 pages
test base rate: 0.597



honest, 5 features                 AUC=0.6170   precision@200=0.785


WITH label-derived column          AUC=0.9922   precision@200=1.000

the leak bought +0.3752 AUC and none of it is real.

what the leaked model leaned on:
  leak_imp_next30      0.457  <- the leak
  imp_90d              0.260
  momentum             0.114
  ctr_m0               0.062
  active_days_m0       0.054
  avg_position_m0      0.053


In [8]:
# Delete it and keep the honest number. This is the one that goes in the report.
model_frame = model_frame.drop(columns=["leak_imp_next30"])

print("dropped leak_imp_next30.")
print(f"columns remaining: {[c for c in model_frame.columns if c.startswith('leak')] or 'no leak columns'}")
print()
print(f"HONEST result, the number I will quote:  AUC = {honest_auc:.4f}")
print(f"discarded leaked result:                 AUC = {leaked_auc:.4f}")
print()
print("what the honest model leans on:")
for name, importance in sorted(zip(FEATURES, honest_model.feature_importances_), key=lambda p: -p[1]):
    print(f"  {name:20s} {importance:.3f}")

dropped leak_imp_next30.
columns remaining: no leak columns

HONEST result, the number I will quote:  AUC = 0.6170
discarded leaked result:                 AUC = 0.9922

what the honest model leans on:
  avg_position_m0      0.264
  imp_90d              0.255
  momentum             0.245
  ctr_m0               0.186
  active_days_m0       0.050


**What that showed.** AUC went from **0.617 to 0.992** by adding one column, and the model put 45% of its importance on it. If I had not known where that column came from, 0.99 is exactly the number I would have been proudest of and most wrong about.

The uncomfortable part is how ordinary the leak looked. `imp_next30` is just an impressions total sitting in a dataframe next to five other impressions totals. Nothing about the column name says "this is the answer". The only thing that saved me is the contract in section 1 saying which window each field is allowed to come from. That is what a data contract is actually for, and it is why I wrote the windows down before building anything.

**The honest number is AUC 0.617 on held-out clients**, and that is what I carry forward. It is a weak-but-real signal, not a triumph, and section 4 says why I would expect it to be weak.

A reproducibility note, because it caught me. I ran this twice and got two different AUCs, 0.6205 then 0.6186, and a third value once sorted. Nothing in the model changed: `random_state` was fixed both times. The cause is that DuckDB scans partitions in parallel and does not promise row order, so the second pull came back ordered differently, which moved which clients `GroupShuffleSplit` put in the test set. I now sort the frame by `(client_hash_id, content_hash_id)` right after the pull, which pins it: three runs, including one from a deliberately shuffled input, now give 0.617030 every time. A result that moves when you re-run it is not a result yet, and I would rather find that here than in Week 7.

One number I am deliberately *not* going to spin: the honest model also scores a precision@200 well above its base rate, and in ML-03 my best rule scored 0.710. It would be easy to write "the model beats the rule". It doesn't, because those two numbers are not comparable: different dataset, different label definition, different eligible population, different K relative to pool size. Comparing them would be exactly the kind of quiet cheat this notebook is about. The rule baseline has to be rebuilt on *this* frame before any comparison is allowed, and that is ML-05's job.

## 4. Data limits

**The named limitation: my slice is 39 clients out of 104, and they are the well-instrumented ones.**

The panel is unbalanced, and not gently. The query below counts it: of 104 clients in `dim_clients`, only **24** have six or more months of GSC history at my cutoff, **37** have no `gsc_data_start` at all, and **10** do not start until after T. Month 2026-03 contains 55 clients, and after I require 50 March impressions and non-null features, **39** survive into my model.

So anything I measure is measured on clients who already had search analytics wired up and running for months. Those are plausibly the more mature, better-resourced sites. A refresh model that works on them is **not** evidence it works on a client three weeks into onboarding, and I should not let anyone read it that way.

Three more limits I want on the record:

- **A 30-day label is short.** Search demand is seasonal and April includes holiday movement. A page flagged as declining may just be past its seasonal peak. I can measure decline, but I cannot separate a page going bad from a topic going quiet, and I will not claim to.
- **Absence is not zero.** The daily fact only accrues from a page's registration day, so a page with no rows in January is not a page with zero January impressions, it is a page nobody was watching yet. My 90-day sums quietly treat those the same, which biases `imp_90d` downward for newer pages.
- **`dim_content` cannot be time-travelled at all.** Section 2 killed the date columns, but the same logic applies to `search_volume`, `word_count` and `backlinks`: they carry no timestamp, so I cannot prove any of them held their current value at T. I used none of them as features, and I would need a dated source before I could.

None of this makes the lane unworkable. It makes the claim narrower: **on GSC-instrumented clients with at least three months of history, these five pre-cutoff signals carry a weak, measurable, directional relationship with next-month demand loss.** That sentence I can defend.

In [9]:
# The panel, counted rather than asserted.
panel = con.sql(f'''
    SELECT
        CASE
            WHEN gsc_data_start IS NULL                    THEN 'no GSC start date at all'
            WHEN gsc_data_start >  DATE '{CUTOFF}'         THEN 'starts after my cutoff'
            WHEN gsc_data_start >  DATE '2026-01-01'       THEN 'under 3 months at T'
            WHEN gsc_data_start >  DATE '2025-09-30'       THEN '3 to 6 months at T'
            ELSE                                                '6+ months at T'
        END                AS history_depth,
        COUNT(*)           AS clients
    FROM {DIM_CLIENTS}
    GROUP BY 1
    ORDER BY clients DESC
''').df()

print("GSC history depth per client, at T:")
print(panel.to_string(index=False))
print()
total_clients = int(panel["clients"].sum())
in_model = model_frame["client_hash_id"].nunique()
print(f"clients in dim_clients:        {total_clients}")
print(f"clients in month=2026-03:      55")
print(f"clients in my model frame:     {in_model}  ({in_model / total_clients * 100:.0f}% of the panel)")
print()
print("so my result describes well-instrumented clients, not the panel. that is the limitation.")

GSC history depth per client, at T:
           history_depth  clients
no GSC start date at all       37
          6+ months at T       24
     under 3 months at T       17
      3 to 6 months at T       16
  starts after my cutoff       10

clients in dim_clients:        104
clients in month=2026-03:      55
clients in my model frame:     39  (38% of the panel)

so my result describes well-instrumented clients, not the panel. that is the limitation.


## What ML-04 changed about my plan

1. **Staleness is gone as a feature.** It was my best rule in ML-03 and it is not available at prediction time on the warehouse. Anything I build now comes from the dated fact table.
2. **The honest baseline is AUC 0.617 on held-out clients**, established before any tuning, so ML-05 has a real number to beat rather than a number to invent.
3. **The 50-impression floor still needs a sensitivity check.** I flagged this in ML-03, carried it here unchanged, and it is now the first thing on the ML-05 list rather than a note I keep repeating.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.